In [2]:
import os

folders = [
    "DocuMind_Django/config",
    "DocuMind_Django/assistant/templates/assistant"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [3]:
files = [
    "manage.py",
    "requirements.txt",
    "build.sh",
    "Procfile",
    ".python-version",
    ".gitignore",
    "config/__init__.py",
    "config/settings.py",
    "config/urls.py",
    "config/wsgi.py",
    "assistant/__init__.py",
    "assistant/apps.py",
    "assistant/views.py",
    "assistant/templates/assistant/home.html",
    "README.md"
]

base = "DocuMind_Django"

for file in files:
    path = os.path.join(base, file)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "a").close()

print("All project files created!")

All project files created!


In [4]:
%%writefile /content/DocuMind_Django/requirements.txt

Django>=5.2,<5.3
gunicorn>=23,<24
whitenoise>=6.9,<7
requests>=2.32,<3
PyMuPDF>=1.26,<2

Overwriting /content/DocuMind_Django/requirements.txt


In [5]:
%%writefile /content/DocuMind_Django/manage.py

import os
import sys

if __name__ == "__main__":
    os.environ.setdefault(
        "DJANGO_SETTINGS_MODULE",
        "config.settings"
    )

    from django.core.management import execute_from_command_line

    execute_from_command_line(sys.argv)

Overwriting /content/DocuMind_Django/manage.py


In [6]:
%%writefile /content/DocuMind_Django/config/settings.py

import os
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent

SECRET_KEY = os.getenv(
    "DJANGO_SECRET_KEY",
    "development-secret-key-change-before-deployment"
)

DEBUG = os.getenv("DEBUG", "True").lower() == "true"

ALLOWED_HOSTS = [
    "localhost",
    "127.0.0.1",
    ".onrender.com",
]

INSTALLED_APPS = [
    "django.contrib.admin",
    "django.contrib.auth",
    "django.contrib.contenttypes",
    "django.contrib.sessions",
    "django.contrib.messages",
    "django.contrib.staticfiles",
    "assistant",
]

MIDDLEWARE = [
    "django.middleware.security.SecurityMiddleware",
    "whitenoise.middleware.WhiteNoiseMiddleware",
    "django.contrib.sessions.middleware.SessionMiddleware",
    "django.middleware.common.CommonMiddleware",
    "django.middleware.csrf.CsrfViewMiddleware",
    "django.contrib.auth.middleware.AuthenticationMiddleware",
    "django.contrib.messages.middleware.MessageMiddleware",
    "django.middleware.clickjacking.XFrameOptionsMiddleware",
]

ROOT_URLCONF = "config.urls"

TEMPLATES = [
    {
        "BACKEND": "django.template.backends.django.DjangoTemplates",
        "DIRS": [],
        "APP_DIRS": True,
        "OPTIONS": {
            "context_processors": [
                "django.template.context_processors.request",
                "django.contrib.auth.context_processors.auth",
                "django.contrib.messages.context_processors.messages",
            ],
        },
    },
]

WSGI_APPLICATION = "config.wsgi.application"

DATABASES = {
    "default": {
        "ENGINE": "django.db.backends.sqlite3",
        "NAME": BASE_DIR / "db.sqlite3",
    }
}

AUTH_PASSWORD_VALIDATORS = []

LANGUAGE_CODE = "en-us"
TIME_ZONE = "Asia/Kolkata"

USE_I18N = True
USE_TZ = True

STATIC_URL = "static/"
STATIC_ROOT = BASE_DIR / "staticfiles"

STORAGES = {
    "default": {
        "BACKEND": "django.core.files.storage.FileSystemStorage",
    },
    "staticfiles": {
        "BACKEND": "whitenoise.storage.CompressedManifestStaticFilesStorage",
    },
}

DEFAULT_AUTO_FIELD = "django.db.models.BigAutoField"

FILE_UPLOAD_MAX_MEMORY_SIZE = 15 * 1024 * 1024
DATA_UPLOAD_MAX_MEMORY_SIZE = 16 * 1024 * 1024

Overwriting /content/DocuMind_Django/config/settings.py


In [7]:
import os

print(os.path.exists(
    "/content/DocuMind_Django/config/settings.py"
))

True


In [8]:
%%writefile /content/DocuMind_Django/config/urls.py

from django.contrib import admin
from django.urls import path
from assistant import views

urlpatterns = [
    path("admin/", admin.site.urls),
    path("", views.home, name="home"),
    path("upload/", views.upload_pdf, name="upload_pdf"),
    path("ask/", views.ask_question, name="ask_question"),
]

Overwriting /content/DocuMind_Django/config/urls.py


In [9]:
%%writefile /content/DocuMind_Django/config/wsgi.py

import os
from django.core.wsgi import get_wsgi_application

os.environ.setdefault(
    "DJANGO_SETTINGS_MODULE",
    "config.settings"
)

application = get_wsgi_application()

Overwriting /content/DocuMind_Django/config/wsgi.py


In [10]:
%%writefile /content/DocuMind_Django/assistant/views.py

import os
import uuid
import math
import pymupdf as fitz
import requests

from django.shortcuts import render, redirect
from django.views.decorators.http import require_POST


# Ollama Cloud configuration
API_KEY = os.getenv("OLLAMA_API_KEY", "").strip()

CHAT_MODEL = os.getenv(
    "OLLAMA_MODEL",
    "gemma4:31b"
)

EMBED_MODEL = os.getenv(
    "OLLAMA_EMBED_MODEL",
    "embeddinggemma"
)

OLLAMA_BASE_URL = os.getenv(
    "OLLAMA_BASE_URL",
    "https://ollama.com/api"
).rstrip("/")


# Temporary document storage
DOCUMENTS = {}


# Call Ollama API
def ollama_api(endpoint, payload):

    if not API_KEY:
        raise RuntimeError(
            "OLLAMA_API_KEY is missing."
        )

    response = requests.post(
        f"{OLLAMA_BASE_URL}/{endpoint}",
        headers={
            "Authorization": f"Bearer {API_KEY}"
        },
        json=payload,
        timeout=180
    )

    if not response.ok:
        raise RuntimeError(
            f"Ollama API Error {response.status_code}: "
            f"{response.text[:500]}"
        )

    return response.json()


# Split document into smaller sections
def split_text(text, size=700, overlap=100):

    step = size - overlap

    return [
        text[i:i + size].strip()
        for i in range(0, len(text), step)
        if text[i:i + size].strip()
    ]


# Generate embeddings
def embed(texts):

    vectors = []

    for start in range(0, len(texts), 8):

        batch = texts[start:start + 8]

        result = ollama_api(
            "embed",
            {
                "model": EMBED_MODEL,
                "input": batch
            }
        )

        vectors.extend(
            result.get("embeddings", [])
        )

    if len(vectors) != len(texts):
        raise RuntimeError(
            "Unexpected number of embeddings received."
        )

    return vectors


# Calculate similarity
def cosine_similarity(a, b):

    dot = sum(
        x * y for x, y in zip(a, b)
    )

    norm_a = math.sqrt(
        sum(x * x for x in a)
    )

    norm_b = math.sqrt(
        sum(y * y for y in b)
    )

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot / (norm_a * norm_b)


# Home page
def home(request):

    document_id = request.session.get("document_id")

    return render(
        request,
        "assistant/home.html",
        {
            "has_document": document_id in DOCUMENTS,

            "document_name": request.session.get(
                "document_name", ""
            ),

            "message": request.session.pop(
                "message", ""
            ),

            "answer": request.session.pop(
                "answer", ""
            ),

            "sources": request.session.pop(
                "sources", []
            ),
        }
    )


# Upload and process PDF
@require_POST
def upload_pdf(request):

    uploaded = request.FILES.get("pdf")

    if not uploaded:

        request.session["message"] = (
            "Please select a PDF file."
        )

        return redirect("home")

    if not uploaded.name.lower().endswith(".pdf"):

        request.session["message"] = (
            "Only PDF files are supported."
        )

        return redirect("home")

    if uploaded.size > 15 * 1024 * 1024:

        request.session["message"] = (
            "PDF size must be below 15 MB."
        )

        return redirect("home")

    try:

        pdf_bytes = uploaded.read()

        with fitz.open(
            stream=pdf_bytes,
            filetype="pdf"
        ) as pdf:

            pages = [
                page.get_text()
                for page in pdf
            ]

        full_text = "\n".join(pages).strip()

        if not full_text:

            raise RuntimeError(
                "No selectable text found. "
                "Scanned PDFs require OCR."
            )

        chunks = split_text(full_text)

        vectors = embed(chunks)

        doc_id = uuid.uuid4().hex

        DOCUMENTS[doc_id] = {
            "chunks": chunks,
            "vectors": vectors
        }

        request.session["document_id"] = doc_id

        request.session["document_name"] = uploaded.name

        request.session["message"] = (
            f"Successfully processed {uploaded.name}. "
            f"{len(pages)} pages and {len(chunks)} sections."
        )

    except Exception as exc:

        request.session["message"] = (
            f"PDF processing failed: {exc}"
        )

    return redirect("home")


# Ask questions about uploaded PDF
@require_POST
def ask_question(request):

    question = request.POST.get(
        "question", ""
    ).strip()

    doc_id = request.session.get("document_id")

    document = DOCUMENTS.get(doc_id)

    if not document:

        request.session["answer"] = (
            "Please upload your PDF again. "
            "The document may have been cleared after a restart."
        )

        return redirect("home")

    if not question:

        request.session["answer"] = (
            "Please enter a question."
        )

        return redirect("home")

    try:

        query_vector = embed([question])[0]

        scored = []

        for index, vector in enumerate(
            document["vectors"]
        ):

            score = cosine_similarity(
                query_vector,
                vector
            )

            scored.append((score, index))

        best = sorted(
            scored,
            reverse=True
        )[:3]

        context = "\n\n".join(
            document["chunks"][index]
            for _, index in best
        )

        prompt = f"""
You are DocuMind, an AI PDF assistant.

Answer the user's question using only the
document context provided below.

If the answer is not available in the document,
clearly say that the document does not contain
the answer.

Do not invent information.

DOCUMENT CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
"""

        result = ollama_api(
            "generate",
            {
                "model": CHAT_MODEL,
                "prompt": prompt,
                "stream": False
            }
        )

        request.session["answer"] = result.get(
            "response",
            "No answer was generated."
        ).strip()

        request.session["sources"] = [
            f"Section {index + 1}"
            for _, index in best
        ]

    except Exception as exc:

        request.session["answer"] = (
            f"Could not generate answer: {exc}"
        )

    return redirect("home")

Overwriting /content/DocuMind_Django/assistant/views.py


In [11]:
%%writefile /content/DocuMind_Django/assistant/apps.py

from django.apps import AppConfig


class AssistantConfig(AppConfig):

    default_auto_field = "django.db.models.BigAutoField"

    name = "assistant"

Overwriting /content/DocuMind_Django/assistant/apps.py


In [12]:
%%writefile /content/DocuMind_Django/config/__init__.py

Overwriting /content/DocuMind_Django/config/__init__.py


In [13]:
%%writefile /content/DocuMind_Django/assistant/__init__.py

Overwriting /content/DocuMind_Django/assistant/__init__.py


In [14]:
%%writefile /content/DocuMind_Django/assistant/templates/assistant/home.html

<!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>DocuMind AI - PDF Assistant</title>

    <style>
        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            font-family: Arial, sans-serif;
            background: #f3f6fb;
            color: #172033;
        }

        .container {
            max-width: 850px;
            margin: 45px auto;
            padding: 20px;
        }

        .header {
            text-align: center;
            margin-bottom: 30px;
        }

        .logo {
            font-size: 48px;
        }

        h1 {
            color: #354fc2;
            font-size: 34px;
            margin: 8px 0;
        }

        .subtitle {
            color: #718096;
        }

        .card {
            background: white;
            padding: 25px;
            margin-bottom: 22px;
            border-radius: 16px;
            box-shadow: 0 5px 20px rgba(0,0,0,0.05);
        }

        h2 {
            font-size: 20px;
            margin-top: 0;
        }

        .form-row {
            display: flex;
            gap: 12px;
            flex-wrap: wrap;
        }

        input[type="file"],
        input[type="text"] {
            flex: 1;
            min-width: 200px;
            padding: 13px;
            border: 1px solid #d6deec;
            border-radius: 9px;
        }

        button {
            background: #405bd5;
            color: white;
            border: none;
            padding: 13px 20px;
            border-radius: 9px;
            cursor: pointer;
            font-weight: bold;
        }

        button:hover {
            background: #3049bb;
        }

        .message {
            background: #eaf1ff;
            border-left: 4px solid #405bd5;
            padding: 15px;
            margin-bottom: 20px;
            border-radius: 7px;
            overflow-wrap: anywhere;
        }

        .answer {
            background: #f7f9ff;
            padding: 18px;
            border-radius: 10px;
            line-height: 1.7;
            white-space: pre-wrap;
            overflow-wrap: anywhere;
        }

        .small {
            color: #77839a;
            font-size: 13px;
        }

        footer {
            text-align: center;
            color: #8993a6;
            font-size: 13px;
            margin-top: 35px;
        }

        @media(max-width: 600px) {
            .container {
                margin: 20px auto;
            }

            .form-row {
                flex-direction: column;
            }

            button {
                width: 100%;
            }
        }
    </style>

</head>

<body>

<div class="container">

    <div class="header">
        <div class="logo">📚</div>

        <h1>DocuMind AI</h1>

        <p class="subtitle">
            Your intelligent PDF reading assistant
        </p>
    </div>


    {% if message %}
        <div class="message">
            {{ message }}
        </div>
    {% endif %}


    <div class="card">

        <h2>📄 Upload your PDF</h2>

        <form
            action="{% url 'upload_pdf' %}"
            method="POST"
            enctype="multipart/form-data">

            {% csrf_token %}

            <div class="form-row">

                <input
                    type="file"
                    name="pdf"
                    accept=".pdf,application/pdf"
                    required>

                <button type="submit">
                    Upload & Process
                </button>

            </div>

        </form>

        {% if has_document %}

            <p class="small">
                Current document:
                <strong>{{ document_name }}</strong>
            </p>

        {% else %}

            <p class="small">
                PDF only · Maximum 15 MB
            </p>

        {% endif %}

    </div>


    <div class="card">

        <h2>💬 Ask your PDF</h2>

        <form
            action="{% url 'ask_question' %}"
            method="POST">

            {% csrf_token %}

            <div class="form-row">

                <input
                    type="text"
                    name="question"
                    placeholder="Ask something about your PDF..."
                    required>

                <button type="submit">
                    Ask DocuMind
                </button>

            </div>

        </form>

    </div>


    {% if answer %}

        <div class="card">

            <h2>🤖 AI Answer</h2>

            <div class="answer">
                {{ answer }}
            </div>

            {% if sources %}

                <p class="small">
                    Sources: {{ sources|join:", " }}
                </p>

            {% endif %}

        </div>

    {% endif %}


    <footer>
        DocuMind AI · Powered by Django and Ollama
    </footer>

</div>

</body>
</html>

Overwriting /content/DocuMind_Django/assistant/templates/assistant/home.html


In [15]:
%%writefile /content/DocuMind_Django/build.sh
#!/usr/bin/env bash
set -o errexit

pip install -r requirements.txt

python manage.py collectstatic --noinput

python manage.py migrate

Overwriting /content/DocuMind_Django/build.sh


In [16]:
%%writefile /content/DocuMind_Django/Procfile
web: gunicorn config.wsgi:application --bind 0.0.0.0:$PORT

Overwriting /content/DocuMind_Django/Procfile


In [17]:
%%writefile /content/DocuMind_Django/.python-version
3.11.11

Overwriting /content/DocuMind_Django/.python-version


In [18]:
%%writefile /content/DocuMind_Django/.gitignore
__pycache__/
*.py[cod]
.venv/
.env
db.sqlite3
staticfiles/
media/

Overwriting /content/DocuMind_Django/.gitignore


In [19]:
import os

for root, dirs, files in os.walk(
    "/content/DocuMind_Django"
):
    for file in files:
        print(os.path.relpath(
            os.path.join(root, file),
            "/content/DocuMind_Django"
        ))

build.sh
Procfile
manage.py
README.md
requirements.txt
.python-version
.gitignore
config/settings.py
config/__init__.py
config/urls.py
config/wsgi.py
assistant/apps.py
assistant/__init__.py
assistant/views.py
assistant/templates/assistant/home.html


In [20]:
!pip install django

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 22.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [django]


In [21]:
import django

print("Django version:", django.get_version())

Django version: 6.1.3


In [22]:
!pip install "Django>=5.2,<5.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 19.1 MB/s  0:00:00
  Attempting uninstall: Django
    Found existing installation: Django 6.1.3
    Uninstalling Django-6.1.3:
      Successfully uninstalled Django-6.1.3


In [1]:
import django
print(django.get_version())

5.2.19


In [2]:
%cd /content/DocuMind_Django
!python manage.py check

/content/DocuMind_Django
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/django/core/checks/urls.py", line 136, in check_custom_error_handlers
    handler = resolver.resolve_error_handler(status_code)
  File "/usr/local/lib/python3.13/dist-packages/django/urls/resolvers.py", line 732, in resolve_error_handler
    callback = getattr(self.urlconf_module, "handler%s" % view_type, None)
                       ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/django/utils/functional.py", line 47, in __get__
    res = instance.__dict__[self.name] = self.func(instance)
                                         ~~~~~~~~~^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/django/urls/resolvers.py", line 711, in urlconf_module
    return import_module(self.urlconf_name)
  File "/usr/lib/python3.13/importlib/__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~

In [3]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 55.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 116.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [PyMuPDF]


In [4]:
import fitz

print("PyMuPDF installed successfully!")
print("Version:", fitz.VersionBind)

PyMuPDF installed successfully!
Version: 2.2


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [5]:
%cd /content/DocuMind_Django
!python manage.py check

/content/DocuMind_Django
/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
System check identified no issues (0 silenced).


In [6]:
%cd /content/DocuMind_Django
!python manage.py check

/content/DocuMind_Django
/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
System check identified no issues (0 silenced).


In [7]:
%cd /content/DocuMind_Django
!python manage.py migrate

/content/DocuMind_Django
/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
Operations to perform:
  Apply all migrations: admin, auth, contenttypes, sessions
Running migrations:
  Applying contenttypes.0001_initial... OK
  Applying auth.0001_initial... OK
  Applying admin.0001_initial... OK
  Applying admin.0002_logentry_remove_auto_add... OK
  Applying admin.0003_logentry_add_action_flag_choices... OK
  Applying contenttypes.0002_remove_content_type_name... OK
  Applying auth.0002_alter_permission_name_max_length... OK
  Applying auth.0003_alter_user_email_max_length... OK
  Applying auth.0004_alter_user_username_opts... OK
  Applying auth.0005_alter_user_last_login_null... OK
  Applying auth.0006_require_contenttypes_0002... OK
  Applying auth.0007_alter_validators_add_error_messages... OK
  Applying auth.0008_alter_user_username_max_length... OK
  Appl

In [10]:
!pip install whitenoise

In [12]:
!pip install pycloudflared

In [13]:
from pycloudflared import try_cloudflare

tunnel_url = try_cloudflare(port=8000)
print("Your DocuMind website link:", tunnel_url)

Download cloudflared...:   0%|          | 0/40334417 [00:00<?, ?it/s]

 * Running on https://venues-ericsson-industry-revisions.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics
Your DocuMind website link: Urls(tunnel='https://venues-ericsson-industry-revisions.trycloudflare.com', metrics='http://127.0.0.1:20241/metrics', process=<Popen: returncode: None args: ['/usr/local/lib/python3.13/dist-packages/pyc...>)


In [14]:
%cd /content/DocuMind_Django

!grep -nE "SECRET_KEY|DEBUG|OLLAMA_API_KEY|os.environ" config/settings.py assistant/views.py

/content/DocuMind_Django
config/settings.py:7:SECRET_KEY = os.getenv(
config/settings.py:8:    "DJANGO_SECRET_KEY",
config/settings.py:12:DEBUG = os.getenv("DEBUG", "True").lower() == "true"
assistant/views.py:13:API_KEY = os.getenv("OLLAMA_API_KEY", "").strip()
assistant/views.py:40:            "OLLAMA_API_KEY is missing."


In [15]:
SECRET_KEY = os.getenv(
    "DJANGO_SECRET_KEY",

_IncompleteInputError: incomplete input (1436223030.py, line 2)

In [17]:
DEBUG = os.getenv("DEBUG", "True").lower() == "true"

NameError: name 'os' is not defined

In [19]:
import os

In [20]:
DEBUG = os.getenv("DEBUG", "True").lower() == "true"
print(DEBUG)

True


In [21]:
import os

os.environ["DEBUG"] = "False"

DEBUG = os.getenv("DEBUG", "True").lower() == "true"

print(DEBUG)

False


In [22]:
%cd /content/DocuMind_Django

!find . -maxdepth 3 -type f

/content/DocuMind_Django
./build.sh
./config/settings.py
./config/__init__.py
./config/__pycache__/wsgi.cpython-313.pyc
./config/__pycache__/settings.cpython-313.pyc
./config/__pycache__/urls.cpython-313.pyc
./config/__pycache__/__init__.cpython-313.pyc
./config/urls.py
./config/wsgi.py
./Procfile
./manage.py
./README.md
./requirements.txt
./assistant/apps.py
./assistant/__init__.py
./assistant/__pycache__/apps.cpython-313.pyc
./assistant/__pycache__/views.cpython-313.pyc
./assistant/__pycache__/__init__.cpython-313.pyc
./assistant/views.py
./db.sqlite3
./.python-version
./.gitignore


In [23]:
import shutil

shutil.make_archive(
    "/content/DocuMind_Django",
    "zip",
    "/content/DocuMind_Django"
)

print("ZIP file created successfully!")

ZIP file created successfully!


In [24]:
from google.colab import files

files.download("/content/DocuMind_Django.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>